# 01 — Data Exploration
**Project:** ViT Reliability & Explainability Under Medical Distribution Shift

**Goal:** Load NIH ChestX-ray14, explore class distribution, visualize samples, confirm data is ready for training.

Run all cells top to bottom every new Colab session.

## 0. Setup — Run every session

In [ ]:
import os

REPO = 'vit-medical-shift'
GITHUB_URL = 'https://github.com/YOUR_USERNAME/vit-medical-shift.git'  # <- change YOUR_USERNAME

if os.path.exists(f'/content/{REPO}'):
    print('Repo exists, pulling latest...')
    os.system(f'git -C /content/{REPO} pull origin main')
else:
    print('Cloning repo...')
    os.system(f'git clone {GITHUB_URL} /content/{REPO}')

import sys
sys.path.insert(0, f'/content/{REPO}')
print('Done!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q timm torchmetrics grad-cam einops
print('Packages ready!')

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name   :', torch.cuda.get_device_name(0))
    print('GPU memory :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Paths

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/data/nih'
CSV_PATH   = f'{DRIVE_ROOT}/Data_Entry_2017.csv'
BBOX_PATH  = f'{DRIVE_ROOT}/BBox_List_2017.csv'
IMG_DIR    = f'{DRIVE_ROOT}/images'

print('CSV exists  :', os.path.exists(CSV_PATH))
print('BBox exists :', os.path.exists(BBOX_PATH))
print('Images dir  :', os.path.exists(IMG_DIR))
if os.path.exists(IMG_DIR):
    print('Images found:', f'{len(os.listdir(IMG_DIR)):,}')

## 2. Load CSV

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(CSV_PATH)
print('Shape:', df.shape)
print('Total images       :', len(df))
print('Unique patients    :', df['Patient ID'].nunique())
print('\nGender distribution:')
print(df['Patient Gender'].value_counts())
df.head()

## 3. Class Distribution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.utils import NIH_LABELS

label_counts = {}
for label in NIH_LABELS:
    label_counts[label] = df['Finding Labels'].str.contains(label).sum()
label_counts['No Finding'] = df['Finding Labels'].str.contains('No Finding').sum()
label_counts = dict(sorted(label_counts.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(list(label_counts.keys()), list(label_counts.values()), color='#378ADD', edgecolor='white')
ax.set_xlabel('Number of images')
ax.set_title('NIH ChestX-ray14 — Class Distribution', fontsize=13, fontweight='bold')
ax.invert_yaxis()
for bar, val in zip(bars, label_counts.values()):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=8)
plt.tight_layout()
os.makedirs(f'/content/{REPO}/results/figures', exist_ok=True)
plt.savefig(f'/content/{REPO}/results/figures/class_distribution.png', dpi=150)
plt.show()

## 4. Multi-label Distribution

In [ ]:
df['n_labels'] = df['Finding Labels'].apply(
    lambda x: 0 if x == 'No Finding' else len(x.split('|'))
)

fig, ax = plt.subplots(figsize=(8, 4))
df['n_labels'].value_counts().sort_index().plot(kind='bar', ax=ax, color='#378ADD', edgecolor='white')
ax.set_xlabel('Number of conditions per image')
ax.set_ylabel('Count')
ax.set_title('Multi-label distribution', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/{REPO}/results/figures/multilabel_dist.png', dpi=150)
plt.show()

## 5. Sample Images (after images are downloaded)

In [ ]:
from PIL import Image

def show_samples(df, img_dir, label=None, n=8, cols=4):
    subset = df[df['Finding Labels'].str.contains(label)].sample(n, random_state=42) if label else df.sample(n, random_state=42)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
    axes = axes.flatten()
    for i, (_, row) in enumerate(subset.iterrows()):
        img_path = os.path.join(img_dir, row['Image Index'])
        if os.path.exists(img_path):
            axes[i].imshow(Image.open(img_path).convert('RGB'), cmap='gray')
        axes[i].set_title(row['Finding Labels'][:25], fontsize=7)
        axes[i].axis('off')
    for j in range(i+1, len(axes)): axes[j].axis('off')
    fig.suptitle(label or 'Random samples', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'/content/{REPO}/results/figures/samples_{label or "random"}.png', dpi=150)
    plt.show()

if os.path.exists(IMG_DIR) and len(os.listdir(IMG_DIR)) > 0:
    show_samples(df, IMG_DIR)
    show_samples(df, IMG_DIR, label='Pneumonia')
    show_samples(df, IMG_DIR, label='Cardiomegaly')
else:
    print('Images not downloaded yet.')
    print('CSV exploration above still works — download images in Colab next.')

## 6. Bounding Boxes (for XAI evaluation)

In [ ]:
bbox_df = pd.read_csv(BBOX_PATH)
print('BBox shape:', bbox_df.shape)
print('\nFindings with bounding boxes:')
print(bbox_df['Finding Label'].value_counts())
bbox_df.head()

## 7. Save Summary & Push to GitHub

In [ ]:
# Save class distribution CSV
summary = pd.DataFrame({
    'label': list(label_counts.keys()),
    'count': list(label_counts.values()),
    'prevalence': [v/len(df) for v in label_counts.values()]
})
os.makedirs(f'/content/{REPO}/results/metrics', exist_ok=True)
summary.to_csv(f'/content/{REPO}/results/metrics/class_distribution.csv', index=False)
print('Saved!')

# Push to GitHub
os.chdir(f'/content/{REPO}')
!git config user.email "your@gwu.edu"    # <- change this
!git config user.name "Sosna Worku"
!git add results/figures/ results/metrics/
!git commit -m "week 1: data exploration figures and metrics"
!git push origin main